# 04. Reshape — трансформация формы данных

ML-модели, API, базы данных — все хотят данные в разном формате.
Этот ноутбук: как менять форму df не теряя данных.

| Операция | Направление | Зачем |
|----------|------------|-------|
| `stack` | wide → long (колонки → уровень индекса) | нормализация, groupby |
| `unstack` | long → wide (индекс → колонки) | feature matrix для ML |
| `pivot` | long → wide (явный контроль) | из raw-таблицы в матрицу |
| `melt` | wide → long | подготовка к groupby, визуализации |
| `pivot_table` | pivot + агрегация | сводная таблица с дублями |
| `crosstab` | частоты / пропорции | анализ категорий |
| MultiIndex | иерархический индекс | структура после groupby / stack |

## 1. Wide vs Long — что это и почему важно

**Wide** (широкий) — каждый тикер в своей колонке. Удобно читать глазами, но неудобно для groupby и ML.

**Long** (длинный / tidy) — одна строка = одно наблюдение. Стандарт для ML-пайплайнов, groupby, seaborn.

```
WIDE:                    LONG:
date    AAPL  GOOG       date        ticker  price
2024-01  180   140  →    2024-01     AAPL    180
2024-02  185   143       2024-01     GOOG    140
                         2024-02     AAPL    185
                         2024-02     GOOG    143
```

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Цены закрытия в широком формате — как приходят из большинства API
dates = pd.date_range('2024-01-01', periods=5, freq='B')
prices_wide = pd.DataFrame({
    'AAPL': [182.5, 183.1, 181.9, 184.2, 185.0],
    'GOOG': [140.2, 141.0, 139.8, 142.1, 143.5],
    'MSFT': [374.1, 376.0, 373.5, 378.2, 380.1],
}, index=dates)
prices_wide.index.name = 'date'

print('WIDE формат (как из API):')
print(prices_wide)

## 2. `stack` — wide → long через MultiIndex

`stack()` «складывает» колонки внутрь индекса — колонки становятся вторым уровнем индекса.
Результат — Series с MultiIndex `(date, ticker)`.

Это стандартный способ нормализовать price matrix перед groupby.

In [ ]:
# stack: колонки (тикеры) уходят в индекс
prices_long = prices_wide.stack()
prices_long.index.names = ['date', 'ticker']
prices_long.name = 'price'

print('LONG формат после stack:')
print(prices_long)
print(f'\nТип: {type(prices_long).__name__}, индекс: {prices_long.index.names}')

# Теперь можно groupby по тикеру
print('\nСредняя цена по тикеру (после stack → groupby):')
print(prices_long.groupby('ticker').mean().round(2))

## 3. `unstack` — long → wide

`unstack()` — обратная операция: уровень индекса разворачивается в колонки.
Это главный инструмент для построения feature matrix после groupby.

После `groupby(['date','ticker']).agg(...)` результат — MultiIndex.
`unstack('ticker')` превращает его в широкую матрицу `date × ticker`.

In [ ]:
# unstack: обратно к широкому формату
prices_back_wide = prices_long.unstack('ticker')
print('После unstack — снова WIDE:')
print(prices_back_wide)

# Практичный пример: дневные доходности → matrica корреляций
returns = prices_wide.pct_change().dropna()
print('\nДневные доходности (wide — удобно для .corr()):')
print(returns.round(4))
print('\nКорреляционная матрица:')
print(returns.corr().round(3))

## 4. `pivot` — long → wide с явным контролем

`pivot(index, columns, values)` — то же что unstack, но для обычного df (не MultiIndex).
Требует уникальности `(index, columns)` — если дубли, нужен `pivot_table`.

Типичный сценарий: сырые данные из БД приходят в long-формате → pivot → feature matrix.

In [ ]:
# Long-формат как из БД или CSV
raw_signals = pd.DataFrame({
    'date':    pd.to_datetime(['2024-01-02']*3 + ['2024-01-03']*3 + ['2024-01-04']*3),
    'ticker':  ['AAPL','GOOG','MSFT'] * 3,
    'signal':  np.round(np.random.uniform(-1, 1, 9), 3),
    'conf':    np.round(np.random.uniform(0.5, 1.0, 9), 2),
})
print('RAW long-формат (из БД):')
print(raw_signals)

# pivot: одна метрика
signal_matrix = raw_signals.pivot(index='date', columns='ticker', values='signal')
signal_matrix.columns.name = None  # убрать лейбл колонок
print('\nPivot → матрица сигналов (date × ticker):')
print(signal_matrix)

# pivot с несколькими values → MultiIndex колонок
full_matrix = raw_signals.pivot(index='date', columns='ticker', values=['signal','conf'])
print('\nPivot с несколькими values — MultiIndex колонок:')
print(full_matrix)

## 5. `melt` — wide → long явно

`melt(id_vars, value_vars)` — явная альтернатива `stack()`.
Указываешь что «держать» как id, остальное складывается в `variable` + `value`.

Чаще используется когда df не имеет удобного индекса (пришёл из CSV).

In [ ]:
# Wide-данные с дополнительной колонкой-метой (источник данных)
portfolio_wide = pd.DataFrame({
    'date':    pd.date_range('2024-01-01', periods=3, freq='B'),
    'source':  ['Bloomberg', 'Bloomberg', 'Reuters'],
    'AAPL':    [182.5, 183.1, 181.9],
    'GOOG':    [140.2, 141.0, 139.8],
    'MSFT':    [374.1, 376.0, 373.5],
})
print('Wide с мета-колонкой:')
print(portfolio_wide)

# melt: date и source — id, тикеры складываем в long
portfolio_long = pd.melt(
    portfolio_wide,
    id_vars=['date', 'source'],    # эти колонки остаются
    value_vars=['AAPL','GOOG','MSFT'],  # эти разворачиваются
    var_name='ticker',
    value_name='price'
)
print('\nПосле melt — long с мета-данными:')
print(portfolio_long.sort_values(['date','ticker']).reset_index(drop=True))

## 6. `pivot_table` — pivot с агрегацией

Если в данных **дубли** (несколько значений на одну комбинацию index×column),
простой `pivot` упадёт с ошибкой. `pivot_table` агрегирует их.

Это Excel-сводная таблица, но в pandas. Очень мощная для разведочного анализа.

In [ ]:
np.random.seed(7)

# Сделки: несколько сделок по одному тикеру в один день (дубли!)
trades = pd.DataFrame({
    'date':    pd.to_datetime(np.random.choice(
        pd.date_range('2024-01-01', periods=3, freq='B'), 12)),
    'ticker':  np.random.choice(['AAPL','GOOG','MSFT'], 12),
    'sector':  '',
    'pnl':     np.round(np.random.normal(0, 500, 12), 0),
    'volume':  np.random.randint(100, 10000, 12),
})
sector_map = {'AAPL': 'Tech', 'GOOG': 'Tech', 'MSFT': 'Tech'}
trades['sector'] = trades['ticker'].map(sector_map)
# Добавим Finance для разнообразия
trades.loc[trades['ticker'] == 'MSFT', 'sector'] = 'Finance'
print('Сделки (есть дубли по date+ticker):')
print(trades.head(8))

# pivot_table: суммарный P&L по дате и тикеру, fill NaN нулём
pnl_table = trades.pivot_table(
    values='pnl',
    index='date',
    columns='ticker',
    aggfunc='sum',
    fill_value=0
)
print('\nP&L по дате × тикеру (pivot_table, aggfunc=sum):')
print(pnl_table)

# Несколько метрик сразу
multi_table = trades.pivot_table(
    values=['pnl', 'volume'],
    index='sector',
    columns=None,
    aggfunc={'pnl': 'sum', 'volume': 'mean'},
)
print('\nP&L (sum) и объём (mean) по сектору:')
print(multi_table.round(1))

## 7. `crosstab` — частоты и пропорции по категориям

`crosstab` — специализированный `pivot_table` для **подсчёта частот**.
По умолчанию считает количество. С `normalize=True` — доли.

Применение в ML: анализ распределения классов, confusion matrix вручную, quality check данных.

In [ ]:
np.random.seed(0)

# Сигналы от разных моделей — хотим посмотреть, как сигналы распределены по секторам
n = 100
analysis = pd.DataFrame({
    'sector': np.random.choice(['Tech', 'Finance', 'Energy'], n, p=[0.5, 0.3, 0.2]),
    'signal': np.random.choice(['buy', 'hold', 'sell'], n, p=[0.4, 0.35, 0.25]),
    'model':  np.random.choice(['ModelA', 'ModelB'], n),
})

# Частоты: сколько каждого сигнала в каждом секторе
print('Частоты сигналов по секторам:')
print(pd.crosstab(analysis['sector'], analysis['signal']))

# Нормализованные по строкам (доля сигналов внутри каждого сектора)
print('\nДоля сигналов внутри сектора (normalize=\'index\'):')
print(pd.crosstab(analysis['sector'], analysis['signal'], normalize='index').round(2))

# С margins — добавить итоговую строку/колонку
print('\nС итогами (margins=True):')
print(pd.crosstab(analysis['sector'], analysis['signal'], margins=True))

## 8. MultiIndex — работа с иерархическим индексом

MultiIndex появляется после `groupby` + `agg`, `stack`, `pivot` с несколькими values.
Ключевые операции: доступ по уровням, `xs`, `reset_index`, `swaplevel`, flatten колонок.

In [ ]:
np.random.seed(5)

# Типичный результат groupby — MultiIndex
portfolio_data = pd.DataFrame({
    'sector':    np.random.choice(['Tech','Finance','Energy'], 30),
    'ticker':    np.random.choice(['AAPL','GOOG','JPM','GS','XOM'], 30),
    'return_d':  np.round(np.random.normal(0, 0.02, 30), 4),
    'volume_m':  np.round(np.abs(np.random.normal(100, 50, 30)), 1),
})

# groupby по двум ключам → MultiIndex в строках
stats = portfolio_data.groupby(['sector','ticker']).agg(
    mean_ret=('return_d', 'mean'),
    vol=('return_d', 'std'),
    avg_volume=('volume_m', 'mean'),
).round(4)
print('Результат groupby — MultiIndex строк:')
print(stats)

# Доступ к срезу по верхнему уровню
print('\nТолько Tech сектор (.loc или .xs):')
print(stats.loc['Tech'])  # или stats.xs('Tech', level='sector')

# reset_index — «распаковать» MultiIndex обратно в колонки
flat = stats.reset_index()
print('\nПосле reset_index (плоский df):')
print(flat)

In [ ]:
# MultiIndex в КОЛОНКАХ — после pivot с несколькими values
# Проблема: колонки ['signal']['AAPL'] неудобны для sklearn

multi_col = raw_signals.pivot(index='date', columns='ticker', values=['signal','conf'])
print('MultiIndex колонок:')
print(multi_col.columns.tolist())

# Flatten: ('signal','AAPL') → 'signal_AAPL'
multi_col.columns = ['_'.join(col) for col in multi_col.columns]
print('\nПосле flatten — плоские имена колонок:')
print(multi_col.columns.tolist())
print(multi_col)

## 9. Итоговый паттерн: raw → ML feature matrix

Реальный пайплайн: данные из API в long-формате → feature matrix для модели.

In [ ]:
np.random.seed(99)

# Сырые данные: каждая строка — один день × тикер × метрика
dates = pd.date_range('2024-01-01', periods=10, freq='B')
tickers = ['AAPL', 'GOOG', 'MSFT', 'JPM']

raw = pd.DataFrame([
    {'date': d, 'ticker': t,
     'close':  round(np.random.uniform(100, 400), 2),
     'volume': int(np.random.uniform(1e6, 1e8)),
     'signal': round(np.random.uniform(-1, 1), 3)}
    for d in dates for t in tickers
])
print(f'RAW: {raw.shape} строк (long)')
print(raw.head(8))

# Шаг 1: pivot → wide (date × ticker для каждой метрики)
close_wide  = raw.pivot(index='date', columns='ticker', values='close')
signal_wide = raw.pivot(index='date', columns='ticker', values='signal')

# Шаг 2: добавить производные фичи
returns_wide = close_wide.pct_change()  # дневная доходность

# Шаг 3: собрать feature matrix с prefixed колонками
close_wide.columns  = [f'close_{t}' for t in close_wide.columns]
signal_wide.columns = [f'signal_{t}' for t in signal_wide.columns]
returns_wide.columns= [f'ret_{t}' for t in returns_wide.columns]

feature_matrix = pd.concat([close_wide, signal_wide, returns_wide], axis=1).dropna()
print(f'\nFeature matrix: {feature_matrix.shape}')
print(feature_matrix.head(3).round(4))